# Diseño SFV — Laboratorio TABAREC (640 Wp)

**Proyecto:** SFV_LABORATORIO  
**Medidor:** HOL-100504639  
**Sistema fijo:** **24 × 640 Wp** = **15.36 kWp** (~21,529 kWh/año)

Incluye: consumo, cobertura vs factura, strings DC, protecciones, conductores AC (25 m) y caída de tensión.


## Registro

| Etapa | Contenido |
|-------|----------|
| 0–4 | Recibo, ascensor, factura |
| 5 | Sistema 15 kWp (decisión proyecto) |
| 5b | Proyección mensual |
| 6 | Strings DC |
| 7 | Conductores/protecciones DC |
| 8 | Alimentador AC 25 m |
| 9 | Resumen |


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import math

MODULO_WP = 640
KWP_OBJETIVO = 15.0
N_PANELES = 24
KWP = 15.36
PROD_ANUAL = 21529

recibo = {
    "medidor": "HOL-100504639",
    "consumo_kwh": 1580,
    "cuv_cop_kwh": 883.72,
    "valor_energia_cop": 1_396_277.60,
    "contribucion_cop": 279_255.52,
    "alumbrado_publico_cop": 214_802.00,
    "total_pagar_cop": 1_890_340,
}


In [ ]:
CUV = recibo['cuv_cop_kwh']
PCT_CONTRIBUCION = recibo['contribucion_cop'] / recibo['valor_energia_cop']
ALUMBRADO_POR_KWH = recibo['alumbrado_publico_cop'] / recibo['consumo_kwh']
costo_total_kwh = CUV * (1 + PCT_CONTRIBUCION) + ALUMBRADO_POR_KWH
FACTOR_FACTURA_TOTAL = costo_total_kwh / CUV
COSTO_ASCENSOR_MES = 400_000
TARIFA_TOTAL_KWH = recibo['total_pagar_cop'] / recibo['consumo_kwh']
KWH_ASCENSOR_MES = round(COSTO_ASCENSOR_MES / TARIFA_TOTAL_KWH, 1)
consumo_mensual = {'Ago':1521,'Sep':1657,'Oct':1572,'Nov':1594,'Dic':1591,'Ene':1555,'Feb':1580}
prom_recibo = sum(consumo_mensual.values()) / len(consumo_mensual)
prom_diseno = prom_recibo - KWH_ASCENSOR_MES
consumo_anual = prom_diseno * 12
FACTOR_NUEVA_CARGA = 1.10
lab_proy = consumo_anual * FACTOR_NUEVA_CARGA
kwh_obj_factura_cero = lab_proy * FACTOR_FACTURA_TOTAL * 1.05 * 1.10
def factura_completa(kwh):
    e = kwh * CUV; c = e * PCT_CONTRIBUCION; a = kwh * ALUMBRADO_POR_KWH
    return dict(kwh=kwh, energia=round(e), contrib=round(c), alumbrado=round(a),
                total_energia=round(e+c), total=round(e+c+a))
f_diseno = factura_completa(prom_diseno)
print(f'Consumo diseño: {prom_diseno:.1f} kWh/mes ({consumo_anual:.0f} kWh/año)')
print(f'Factura diseño: ${f_diseno["total"]:,}/mes')
print(f'Generación req. factura $0: {kwh_obj_factura_cero:,.0f} kWh/año')
print(f'Sistema instalado ({KWP} kWp): {PROD_ANUAL:,} kWh/año → cobertura {PROD_ANUAL/kwh_obj_factura_cero*100:.1f}% del objetivo')


---
## Etapa 5 — Sistema **15 kWp** (decisión de proyecto)

Potencia fija: **24 paneles × 640 Wp = 15.36 kWp**. Inversores: **2 × SUN2000-8K-LC0**.


In [ ]:
print('='*60)
print(f'SISTEMA — {N_PANELES} × {MODULO_WP} Wp = {KWP} kWp')
print('='*60)
print(f'Producción estimada:  {PROD_ANUAL:,} kWh/año')
print(f'Consumo proyectado:   {lab_proy:,.0f} kWh/año')
print(f'Objetivo factura $0:  {kwh_obj_factura_cero:,.0f} kWh/año')
print(f'Cobertura generación: {PROD_ANUAL/kwh_obj_factura_cero*100:.1f} %')
print(f'Autoconsumo directo:  ~{min(PROD_ANUAL, lab_proy):,.0f} kWh/año')


---
## Etapa 5b — Proyección mensual


In [ ]:
FACTOR_GEN_MES = [0.92, 0.95, 1.0, 1.02, 1.05, 1.08, 1.1, 1.08, 1.02, 0.98, 0.92, 0.88]
meses_cal = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
consumo_mes = prom_diseno * FACTOR_NUEVA_CARGA
gen_mes_base = PROD_ANUAL / 12
filas_mes = []
for i, mes in enumerate(meses_cal):
    kwh_cons = consumo_mes
    kwh_gen = gen_mes_base * FACTOR_GEN_MES[i]
    autoconsumo = min(kwh_cons, kwh_gen)
    excedente = max(0, kwh_gen - kwh_cons)
    f_sin = factura_completa(kwh_cons)
    energia_cubierta = autoconsumo * CUV
    contrib_cubierta = energia_cubierta * PCT_CONTRIBUCION
    alumbrado_cubierto = min(excedente * CUV, kwh_cons * ALUMBRADO_POR_KWH)
    factura_restante = max(0, f_sin['total'] - energia_cubierta - contrib_cubierta - alumbrado_cubierto)
    filas_mes.append({'Mes':mes,'Consumo':round(kwh_cons),'Generación':round(kwh_gen),
        'Factura sin SFV':round(f_sin['total']),'Factura con SFV':round(factura_restante)})
df_mes = pd.DataFrame(filas_mes)
print('PROYECCIÓN MENSUAL — 24 × 640 Wp\n')
print(df_mes.to_string(index=False))
print(f"\nPromedio factura con SFV: ${df_mes['Factura con SFV'].mean():,.0f}/mes")


---
## Etapa 6 — Strings DC

| Inversor | Paneles | Strings |
|----------|---------|--------|
| 2 × SUN2000-8K-LC0 | **{N_LAB} × 640 Wp** | 12 + 12 paneles |


In [ ]:
panel = {'marca': 'Por definir — TOPCon 640 Wp', 'potencia_wp': 640, 'voc': 44.5, 'vmp': 37.5, 'isc': 17.9, 'imp': 16.95, 'coef_voc': 0.0025, 'coef_vmp': -0.004, 'area_m2': 3.094}
INV_8K = dict(modelo='SUN2000-8K-LC0', pac_w=8000, pv_max_wp=12000, mppt=3,
            mppt_vmin=40, mppt_vmax=560, voc_max=600, icc_max=16, isc_max=20)
N_INV = 2
paneles_por_inv = [N_PANELES//2, N_PANELES - N_PANELES//2]

def disenar_strings(n, n_mppt=3):
    b, r = divmod(n, n_mppt)
    return [b+1]*r + [b]*(n_mppt-r)

def verificar_string(n, inv=INV_8K, t_frio=-10, t_cal=60):
    voc = n*panel['voc']*(1+panel['coef_voc']*(25-t_frio))
    vmp = n*panel['vmp']*(1+panel['coef_vmp']*(t_cal-25))
    imp = panel['imp']
    return dict(n=n, voc_frio=round(voc,1), vmp_cal=round(vmp,1), imp=imp,
        ok_voc=voc<=inv['voc_max'], ok_vmin=vmp>=inv['mppt_vmin'],
        ok_vmax=voc<=inv['mppt_vmax'], ok_i=imp<=inv['icc_max'])

config = []
for idx, n_p in enumerate(paneles_por_inv, 1):
    strings = disenar_strings(n_p, INV_8K['mppt'])
    for j, ns in enumerate(strings, 1):
        r = verificar_string(ns)
        r['inversor'] = idx; r['mppt'] = j
        config.append(r)

df_str = pd.DataFrame(config)
print('STRINGS POR INVERSOR\n')
for idx in [1,2]:
    sub = df_str[df_str['inversor']==idx]
    st = ' + '.join(str(x) for x in disenar_strings(paneles_por_inv[idx-1]))
    print(f'Inversor {idx} ({paneles_por_inv[idx-1]} paneles): {st}')
    for _, r in sub.iterrows():
        opt = 'OK' if r['ok_i'] else 'OPTIMIZADOR'
        print(f"  MPPT{int(r['mppt'])}: {int(r['n'])} ser  Voc_frío={r['voc_frio']}V  Vmp_cal={r['vmp_cal']}V  Imp={r['imp']}A  [{opt}]")
if panel['imp'] > INV_8K['icc_max']:
    print(f"\n⚠ Imp ({panel['imp']} A) > 16 A/MPPT → {N_PANELES}× SUN2000-450W-P2")


---
## Etapas 7–8 — Conductores DC y AC (25 m)


In [ ]:
L_DC_M = 40
RHO_CU = 0.01724
DV_MAX_DC = 0.03
FACTOR_I_DC = 1.25
calibres_mm2 = [2.5, 4, 6, 10, 16, 25, 35, 50]
ampacidad_cu_75c = {2.5:24, 4:32, 6:41, 10:57, 16:76, 25:101, 35:125, 50:151}

def calibre_dc(i_req, v_string, l_m=L_DC_M):
    for s in calibres_mm2:
        dv = 2 * l_m * i_req * RHO_CU / s
        if dv / v_string <= DV_MAX_DC and ampacidad_cu_75c[s] >= i_req:
            return s, round(dv,2), round(dv/v_string*100,2)
    s = calibres_mm2[-1]
    dv = 2 * l_m * i_req * RHO_CU / s
    return s, round(dv,2), round(dv/v_string*100,2)

filas_dc = []
for _, r in df_str.iterrows():
    isc_str = panel['isc']
    i_dis = FACTOR_I_DC * isc_str
    s, dv, dv_pct = calibre_dc(i_dis, r['vmp_cal'])
    filas_dc.append({'Inversor': f"Inv{int(r['inversor'])} MPPT{int(r['mppt'])}", 'Paneles': int(r['n']),
        'I diseño (A)': round(i_dis,1), 'Fusible DC (A)': math.ceil(isc_str*1.25*10)/10,
        'Cable mm²': s, 'ΔV (%)': dv_pct})
print('CONDUCTORES DC\n', pd.DataFrame(filas_dc).to_string(index=False))

L_AC_M = 25
V_AC, ETA_INV, DV_MAX_AC, FP = 220, 0.98, 0.03, 1.0
filas_ac = []
for idx, pac in enumerate([INV_8K['pac_w']]*N_INV, 1):
    i_nom = pac / (V_AC * ETA_INV * FP)
    i_dis = i_nom * 1.25
    breaker = math.ceil(i_dis / 5) * 5
    for s in calibres_mm2:
        dv = 2 * L_AC_M * i_nom * RHO_CU / s
        if dv / V_AC <= DV_MAX_AC and ampacidad_cu_75c[s] >= i_dis:
            s_sel, dv_p = s, round(dv/V_AC*100,2); break
    filas_ac.append({'Circuito': f'Inversor {idx}', 'Pac (kW)': pac/1000, 'I nom (A)': round(i_nom,1),
        'Breaker (A)': int(breaker), 'Cable mm²': s_sel, 'ΔV (%)': dv_p})
print('\nALIMENTADOR AC 25 m\n', pd.DataFrame(filas_ac).to_string(index=False))


### Resumen

| Concepto | Valor |
|----------|-------|
| Paneles | **24 × 640 Wp** |
| kWp | **15.36** |
| Inversores | **2 × SUN2000-8K-LC0** |
| Producción est. | ~21,529 kWh/año |
| Optimizadores | Sí (Imp > 16 A) |
| AC | 25 m, 2×50 A, 10 mm² |
